<a href="https://colab.research.google.com/github/wannasmile/colab_code_note/blob/main/PytorchMain01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 数据读入

PyTorch数据读入是通过Dataset+DataLoader的方式完成的，Dataset定义好数据的格式和数据变换形式，DataLoader用iterative的方式不断读入批次数据。


In [1]:
import datasets

bio_mqm_dataset = datasets.load_dataset('zouhar/bio-mqm-dataset')

print("Dataset loaded successfully:")
print(bio_mqm_dataset)

if 'train' in bio_mqm_dataset:
    print("\nFirst few examples from the training split:")
    print(bio_mqm_dataset['train'][0])
else:
    print("\nNo 'train' split found. Displaying first example from the first available split:")
    first_split_name = list(bio_mqm_dataset.keys())[0]
    print(bio_mqm_dataset[first_split_name][0])

README.md:   0%|          | 0.00/2.79k [00:00<?, ?B/s]

dev.jsonl: reconstructing file:   0%|          |  0.00B / 13.8MB            

dev.jsonl: downloading bytes:           |  0.00B            

test.jsonl: reconstructing file:   0%|          |  0.00B / 49.2MB            

test.jsonl: downloading bytes:           |  0.00B            

Generating validation split:   0%|          | 0/12725 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/49448 [00:00<?, ? examples/s]

Dataset loaded successfully:
DatasetDict({
    validation: Dataset({
        features: ['src', 'tgt', 'ref', 'system', 'lang_src', 'lang_tgt', 'annotator', 'errors_src', 'errors_tgt', 'doc_id'],
        num_rows: 12725
    })
    test: Dataset({
        features: ['src', 'tgt', 'ref', 'system', 'lang_src', 'lang_tgt', 'annotator', 'errors_src', 'errors_tgt', 'doc_id'],
        num_rows: 49448
    })
})

No 'train' split found. Displaying first example from the first available split:
{'src': 'Three cases of cervicofacial NF are presented in this case report.', 'tgt': 'In diesem Fallbericht werden drei Fälle von zervikofazialem NF vorgestellt.', 'ref': ['In diesem Fallbericht werden drei Fälle von zervikofazialer NF dargestellt.', 'In diesem Fallbericht werden drei Fälle von zervikofazialer NF dargestellt.'], 'system': 'HuaweiTSC_run1', 'lang_src': 'en', 'lang_tgt': 'de', 'annotator': 'RH2/ende', 'errors_src': [], 'errors_tgt': [{'term': 'zervikofazialem', 'startIndex': 44, 'endIndex': 5

In [2]:
print("\nFirst 5 entries from the 'validation' split:")
display(bio_mqm_dataset['validation'].select(range(5)))


First 5 entries from the 'validation' split:


Dataset({
    features: ['src', 'tgt', 'ref', 'system', 'lang_src', 'lang_tgt', 'annotator', 'errors_src', 'errors_tgt', 'doc_id'],
    num_rows: 5
})

In [3]:
# Define a local path to save the dataset
local_path = "./local_bio_mqm_dataset"

# Save the entire DatasetDict to the local path
bio_mqm_dataset.save_to_disk(local_path)

print(f"Dataset saved to: {local_path}")

Saving the dataset (0/1 shards):   0%|          | 0/12725 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/49448 [00:00<?, ? examples/s]

Dataset saved to: ./local_bio_mqm_dataset


In [4]:
print("\nData structure of the 'validation' split (features):")
print(bio_mqm_dataset['validation'].features)


Data structure of the 'validation' split (features):
{'src': Value('string'), 'tgt': Value('string'), 'ref': List(Value('string')), 'system': Value('string'), 'lang_src': Value('string'), 'lang_tgt': Value('string'), 'annotator': Value('string'), 'errors_src': List({'term': Value('string'), 'startIndex': Value('int64'), 'endIndex': Value('int64'), 'error_category': Value('string'), 'error_subcategory': Value('string'), 'severity': Value('string')}), 'errors_tgt': List({'term': Value('string'), 'startIndex': Value('int64'), 'endIndex': Value('int64'), 'error_category': Value('string'), 'error_subcategory': Value('string'), 'severity': Value('string')}), 'doc_id': Value('string')}


In [5]:
import torch # 导入 PyTorch 深度学习核心库
import datasets # 导入 Hugging Face 数据集加载库
from torch.utils.data import Dataset, DataLoader, random_split # 导入 PyTorch 数据处理核心工具
from transformers import AutoTokenizer # 导入 Hugging Face 分词器工具

# --- PyTorch 2.x 计算设备自动检测与配置 ---
# 优先选择 NVIDIA CUDA GPU，其次选择 Apple Silicon MPS 加速，最后回退到 CPU
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"当前计算设备: {device}")

# --- 深度学习模型训练核心超参数设置 ---
BATCH_SIZE = 16       # 批次大小：每次送入神经网络计算的文本对数量 (矩阵第一维 Batch Size = 16)
LR = 2e-5             # 初始学习率：BERT / Transformer 预训练模型微调时最常用的经典学习率
EPOCHS = 5            # 训练轮数：遍历完整训练集的总次数
MAX_LEN = 128         # 最大序列长度：设定拼接后文本的最大 Token 数量 (Seq_Len = 128)

当前计算设备: cpu


In [6]:
class BioMQMPyTorchDataset(Dataset): # 继承 PyTorch 的 Dataset 基类
    def __init__(self, hf_dataset, tokenizer_name='bert-base-multilingual-cased', max_len=128):
        """
        Args:
            hf_dataset: Hugging Face 的 Dataset 分片 (如 bio_mqm_dataset['validation'])
            tokenizer_name: 预训练多语言分词器名称 (支持英德等跨语言文本编码)
            max_len: 文本最大的 Token 序列长度
        """
        self.hf_dataset = hf_dataset # 内部保存 Hugging Face 数据集对象
        # 加载支持多语言的分词器，负责把词汇转换为数字 ID
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
        self.max_len = max_len # 保存最大的 Token 限制长度

        # 定义 MQM 标准的错误严重程度扣分权重 (用于计算连续回归标签)
        self.severity_weights = {
            'Minor': 1.0,     # 轻微错误扣 1 分
            'Major': 5.0,     # 重大错误扣 5 分
            'Critical': 25.0  # 严重错误扣 25 分
        }

    def __len__(self):
        # 返回当前数据集的总条数 (例如 12725)
        return len(self.hf_dataset)

    def __getitem__(self, idx):
        """
        Args:
            idx (int): 数据索引，DataLoader 运行时会自动传入 0, 1, 2...
        """
        # 1. 获取指定索引的条目字典
        item = self.hf_dataset[idx]

        # 2. 提取源语言文本 (src) 与 翻译目标文本 (tgt)
        src_text = str(item['src']) # 源语言文本 (英语)
        tgt_text = str(item['tgt']) # 目标翻译文本 (德语)

        # 3. 解析 errors_tgt 字段，计算该句译文的 MQM 扣分总和 (作为回归目标 Label)
        errors = item.get('errors_tgt', []) # 获取译文错误列表
        mqm_penalty = 0.0 # 初始化总扣分为 0
        for err in errors: # 遍历当前句子中的每一个错误项
            severity = err.get('severity', 'Minor') # 获取错误的严重级别，默认为 Minor
            mqm_penalty += self.severity_weights.get(severity, 1.0) # 根据级别累加扣分

        # 4. 使用 Tokenizer 将 (src_text, tgt_text) 文本对转成数字张量
        encoding = self.tokenizer(
            src_text,                # 文本 1：原文
            tgt_text,                # 文本 2：译文 (Tokenizer 会自动用特殊字符 [SEP] 隔开两句)
            add_special_tokens=True, # 自动添加 [CLS] 开头标识与 [SEP] 分隔标识
            max_length=self.max_len, # 限制最大长度为 128
            padding='max_length',    # 长度不足 128 则用 0 ([PAD]) 在尾部补齐
            truncation=True,         # 超出 128 则截断
            return_tensors='pt'      # 直接返回 PyTorch Tensor 格式
        )

        # 5. 调整维度：Tokenizer 返回的维度是 [1, max_len]，使用 squeeze(0) 挤压掉第 0 维，得到一维向量 [max_len]
        input_ids = encoding['input_ids'].squeeze(0)          # 输出维度: [128]
        attention_mask = encoding['attention_mask'].squeeze(0)# 输出维度: [128]

        # 6. 将标量扣分转化为 PyTorch 浮点数张量
        label = torch.tensor(mqm_penalty, dtype=torch.float)  # 输出维度: [] (0维标量)

        # 7. 打包返回一个字典，供后续网络模型输入
        return {
            'input_ids': input_ids,       # 文本对应的 Token ID 向量，维度: [128]
            'attention_mask': attention_mask, # 标注哪些是真实词汇(1)、哪些是填充(0)，维度: [128]
            'label': label                # 翻译质量扣分标签，维度: []
        }

In [7]:
# 1. 载入原始 Hugging Face 数据集[cite: 1]
raw_dataset = datasets.load_dataset('zouhar/bio-mqm-dataset')

# 2. 实例化我们自定义的 BioMQMPyTorchDataset 数据集类
full_dataset = BioMQMPyTorchDataset(raw_dataset['validation'], max_len=MAX_LEN)

# 3. 按 8:2 的比例随机划分训练集 (约 10,180 条) 和 验证集 (约 2,545 条)
train_size = int(0.8 * len(full_dataset)) # 计算训练集数量
val_size = len(full_dataset) - train_size  # 计算验证集数量
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size]) # 随机切分

# 4. 构建训练集 DataLoader
train_loader = DataLoader(
    train_dataset,          # 训练集数据源
    batch_size=BATCH_SIZE,  # 每个批次提取 16 条数据
    shuffle=True,           # 训练阶段打乱顺序，防止模型记忆顺序
    drop_last=True,         # 若最后剩余不足 16 条则丢弃，确保 Batch 矩阵维度一致
    num_workers=0           # 读取数据的子进程数 (Windows 下建议设为 0)
)

# 5. 构建验证集 DataLoader
val_loader = DataLoader(
    val_dataset,            # 验证集数据源
    batch_size=BATCH_SIZE,  # 验证阶段保持同样的批次大小
    shuffle=False,          # 验证阶段不打乱数据顺序
    drop_last=False,        # 验证阶段保留尾部全部样本
    num_workers=0
)

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

In [8]:
# 使用 iter() 将 DataLoader 转为迭代器，使用 next() 提取第一个 Batch 的数据
batch_data = next(iter(train_loader))

# 提取张量并移动到计算设备 (GPU / MPS / CPU) 上
input_ids = batch_data['input_ids'].to(device)         # 输入 Token ID 矩阵
attention_mask = batch_data['attention_mask'].to(device) # 注意力掩码矩阵
labels = batch_data['label'].to(device)                 # 对应批次的 MQM 扣分标签

# --- 打印维度校验信息 (深度学习调试的关键步骤) ---
# input_ids 维度为 [Batch_Size, Max_Len] -> 即 [16, 128]
print(f"批次 input_ids 张量维度: {input_ids.shape}")

# attention_mask 维度为 [Batch_Size, Max_Len] -> 即 [16, 128]
print(f"批次 attention_mask 张量维度: {attention_mask.shape}")

# labels 维度为 [Batch_Size] -> 即 [16]
print(f"批次 labels 标签张量维度: {labels.shape}")

# 确认张量所在的物理计算设备
print(f"张量所在设备: {input_ids.device}")

批次 input_ids 张量维度: torch.Size([16, 128])
批次 attention_mask 张量维度: torch.Size([16, 128])
批次 labels 标签张量维度: torch.Size([16])
张量所在设备: cpu


# 模型构建

In [9]:
import torch
from torch import nn

class NLP_MLP(nn.Module):
    # 声明带有模型参数的层：嵌入层与全连接层
    def __init__(self, vocab_size=119547, embed_dim=256, hidden_dim=128):
        # 调用父类 Module 的构造函数进行核心初始化，这是所有 PyTorch 模型必写的标准开头
        super(NLP_MLP, self).__init__()

        # 1. 词嵌入层 (Embedding)：将离散的文字ID转换为连续的稠密向量
        # 维度变换: 输入 [Batch, Seq_Len] -> 输出 [Batch, Seq_Len, embed_dim]
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_dim)

        # 2. 隐藏层 (Linear)：全连接网络提取特征
        # 注意：我们会先将文本序列的所有词向量求平均，再送入全连接层
        # 维度变换: 输入 [Batch, embed_dim] -> 输出 [Batch, hidden_dim]
        self.hidden = nn.Linear(in_features=embed_dim, out_features=hidden_dim)

        # 3. 激活函数 (ReLU)：加入非线性表达能力
        self.act = nn.ReLU()

        # 4. 输出层 (Linear)：回归任务，输出一个标量（翻译质量扣分）
        # 维度变换: 输入 [Batch, hidden_dim] -> 输出 [Batch, 1]
        self.output = nn.Linear(in_features=hidden_dim, out_features=1)

    # 定义模型的前向计算逻辑，即输入文本张量 x 后，数据在网络中如何流转
    def forward(self, x):
        # x 的初始维度: [Batch, Seq_Len] (例如 [16, 128])

        # 经过嵌入层，维度变为: [16, 128, 256]
        embedded = self.embedding(x)

        # 沿着序列长度维度 (dim=1) 求平均，将整句话压缩为一个向量
        # 维度变为: [16, 256]
        sentence_vector = embedded.mean(dim=1)

        # 经过隐藏层和激活函数，维度变为: [16, 128]
        h = self.act(self.hidden(sentence_vector))

        # 经过输出层得到打分，维度变为: [16, 1]
        out = self.output(h)

        # 将 [16, 1] 降维成 [16]，与 DataLoader 中的 labels 维度对齐
        return out.squeeze(-1)

In [10]:
# 模拟 DataLoader 取出的一个 Batch 的 input_ids (Batch_size=16, Max_Len=128)
# 词汇表大小默认设置多语言 BERT 的 119547，所以随机生成 0~119546 的整数 ID
X = torch.randint(low=0, high=119547, size=(16, 128))

# 实例化模型对象
net = NLP_MLP()
print("网络结构：\n", net)

# 前向计算：调用 net(X) 本质上是触发了类的 __call__，进而执行 forward 函数
predictions = net(X)
print("\n预测打分结果 (维度 {}):".format(predictions.shape))
print(predictions)

网络结构：
 NLP_MLP(
  (embedding): Embedding(119547, 256)
  (hidden): Linear(in_features=256, out_features=128, bias=True)
  (act): ReLU()
  (output): Linear(in_features=128, out_features=1, bias=True)
)

预测打分结果 (维度 torch.Size([16])):
tensor([ 0.0174,  0.0380,  0.0294,  0.0434,  0.0344,  0.0218,  0.0249,  0.0137,
         0.0146, -0.0265, -0.0018,  0.0268,  0.0239,  0.0163,  0.0251,  0.0157],
       grad_fn=<SqueezeBackward1>)


In [11]:
class SequenceMeanCenteredLayer(nn.Module):
    def __init__(self):
        # 初始化父类
        super(SequenceMeanCenteredLayer, self).__init__()

    def forward(self, x):
        # 输入 x 维度假设为 [Batch, Seq_Len, Embed_Dim]
        # 计算特征维度 (dim=-1) 的均值，并保持维度数量不变 (keepdim=True)
        mean_val = x.mean(dim=-1, keepdim=True)
        # 将原始输入减去均值进行中心化
        return x - mean_val

In [12]:
class MultiExpertDense(nn.Module):
    def __init__(self, embed_dim=256):
        super(MultiExpertDense, self).__init__()
        # ParameterDict 允许像字典一样管理带名字的权重矩阵
        self.experts = nn.ParameterDict({
            'expert_grammar': nn.Parameter(torch.randn(embed_dim, embed_dim)),
            'expert_vocab': nn.Parameter(torch.randn(embed_dim, embed_dim))
        })
        # 可以随时动态追加新参数
        self.experts.update({'expert_fluency': nn.Parameter(torch.randn(embed_dim, embed_dim))})

    def forward(self, x, task='expert_grammar'):
        # 根据任务类型，选用不同的权重矩阵进行矩阵乘法 (torch.matmul)
        # x 维度 [Batch, embed_dim], 选中权重维度 [embed_dim, embed_dim] -> 结果 [Batch, embed_dim]
        return torch.matmul(x, self.experts[task])

In [13]:
# 定义一维卷积网络测试
# 输入维度要求: [Batch, Channels, Seq_Len]
# 在NLP中相当于: [批量大小, 词向量维度, 句子长度]
conv1d = nn.Conv1d(in_channels=256, out_channels=128, kernel_size=3, padding=1)

# 模拟输入 [Batch=16, Embed_Dim=256, Seq_len=128]
X_text = torch.rand(16, 256, 128)
output = conv1d(X_text)

# 打印输出维度: 填充 padding=1, 窗口大小 3 保证了序列长度 128 不变，通道数变为 128
print(output.shape) # 输出 torch.Size([16, 128, 128])

torch.Size([16, 128, 128])


In [14]:
import torch.nn.functional as F

class TextCNNRegressor(nn.Module):
    def __init__(self, vocab_size=119547, embed_dim=256):
        super(TextCNNRegressor, self).__init__()
        # 1. 词嵌入层
        self.embedding = nn.Embedding(vocab_size, embed_dim)

        # 2. 三组不同感受野的卷积层 (模拟提取 2-gram, 3-gram, 4-gram 特征)
        # in_channels 对应 embed_dim，out_channels 是提取的特征数
        self.conv2 = nn.Conv1d(embed_dim, 100, kernel_size=2)
        self.conv3 = nn.Conv1d(embed_dim, 100, kernel_size=3)
        self.conv4 = nn.Conv1d(embed_dim, 100, kernel_size=4)

        # 3. Dropout层：随机丢弃 50% 神经元，防止模型过拟合
        self.dropout = nn.Dropout(0.5)

        # 4. 全连接回归输出层：输入是三种卷积核拼接后的特征总数 (100+100+100=300)
        self.fc = nn.Linear(300, 1)

    def forward(self, x):
        # 输入 x: [Batch, Seq_Len] (e.g., [16, 128])
        # 嵌入层输出: [16, 128, 256]
        embedded = self.embedding(x)

        # PyTorch 的 Conv1d 要求的输入是 [Batch, Channels, Seq_Len]
        # 需要用 permute 交换维度 1 和 2，转换后维度: [16, 256, 128]
        embedded = embedded.permute(0, 2, 1)

        # 通过卷积和 ReLU 激活函数
        # conv2 输出: [16, 100, 127] (公式: L_out = 128 - 2 + 1)
        c2 = F.relu(self.conv2(embedded))
        # conv3 输出: [16, 100, 126] (公式: L_out = 128 - 3 + 1)
        c3 = F.relu(self.conv3(embedded))
        # conv4 输出: [16, 100, 125] (公式: L_out = 128 - 4 + 1)
        c4 = F.relu(self.conv4(embedded))

        # 全局最大池化层 (Max Pooling 1D)，在序列长度维度 (dim=2) 上提取最大值
        # 压缩掉长度维度，每个 c 变成 [16, 100]
        p2 = F.max_pool1d(c2, c2.shape[2]).squeeze(2)
        p3 = F.max_pool1d(c3, c3.shape[2]).squeeze(2)
        p4 = F.max_pool1d(c4, c4.shape[2]).squeeze(2)

        # 将提取的三个 [16, 100] 的特征在列维度 (dim=1) 进行拼接
        # 拼接后维度: [16, 300]
        cat_features = torch.cat((p2, p3, p4), dim=1)

        # Dropout 操作
        dropped = self.dropout(cat_features)

        # 全连接层映射到标量得分，输出 [16, 1]，然后 squeeze 为 [16]
        output = self.fc(dropped).squeeze(-1)
        return output

# 实例化并打印模型结构
text_cnn = TextCNNRegressor()
print(text_cnn)

TextCNNRegressor(
  (embedding): Embedding(119547, 256)
  (conv2): Conv1d(256, 100, kernel_size=(2,), stride=(1,))
  (conv3): Conv1d(256, 100, kernel_size=(3,), stride=(1,))
  (conv4): Conv1d(256, 100, kernel_size=(4,), stride=(1,))
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=300, out_features=1, bias=True)
)


In [15]:
class MiniTransformerRegressor(nn.Module):
    def __init__(self, vocab_size=119547, embed_dim=256, num_heads=8, num_layers=2):
        super(MiniTransformerRegressor, self).__init__()
        # 1. 词嵌入层
        self.embedding = nn.Embedding(vocab_size, embed_dim)

        # 2. 定义单层 Transformer 编码器 (包含多头自注意力与前馈神经网络)
        # batch_first=True 表示输入张量的第一个维度是 Batch，符合我们 [16, 128, 256] 的习惯
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=512,
            batch_first=True
        )

        # 3. 堆叠多层 Transformer 编码器
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # 4. 回归输出层
        self.fc = nn.Linear(embed_dim, 1)

    def forward(self, x, mask=None):
        # 输入 x: [Batch, Seq_Len] -> [16, 128]

        # 1. 映射为连续向量: [16, 128, 256]
        embedded = self.embedding(x)

        # 2. 送入 Transformer 进行全局自注意力计算
        # src_key_padding_mask 用于告诉模型忽略补齐的 [PAD] 字符 (DataLoader 传入的 attention_mask)
        # 输出维度不变: [16, 128, 256]
        encoded = self.transformer_encoder(embedded, src_key_padding_mask=mask)

        # 3. 提取句首 [CLS] 向量 (索引 0) 作为整个序列的语义表示
        # 维度提取: [16, 256]
        cls_token = encoded[:, 0, :]

        # 4. 全连接层预测扣分: [16]
        output = self.fc(cls_token).squeeze(-1)
        return output

# 实例化模型
transformer_model = MiniTransformerRegressor()

# 验证前向传播尺寸
dummy_input = torch.randint(0, 1000, (16, 128))
# 模拟 DataLoader 的 attention_mask (1表示真实词，0表示填充，Transformer 里需要取反后转为布尔值传递)
dummy_mask = torch.zeros((16, 128), dtype=torch.bool)

out = transformer_model(dummy_input, mask=dummy_mask)
print("\nMini-Transformer 预测输出维度:", out.shape) # 预期输出: torch.Size([16])


Mini-Transformer 预测输出维度: torch.Size([16])
